## Experiment Ariadne

Aim is to correlate OOD-deltas with ID-deltas. For the OOD-Date, we'll need to follow the general recipe for augmentation: filter for ground truths, then augment the ground truths with the behaviour of choice.

I'm going to introduce an improvement to the augmentation prompt strategy: To reduce noise, we're going to add context to the augmentation prompt
to only augment the trace *in case the augmentation is reasonable*. We're going to be extra careful in inserting the error and we'll follow
the general prompting strategy of [previous works](https://openreview.net/forum?id=IUdJM5HJySV).

#### 09.12 Update
- removing samples which contain "```python" keyword, as python-code hallucination seems to be prevalent in Qwen2.5-7B generated answers.
- removing answers which aren't parsable, i.e. which do not contain "\\boxed{}" in last 300 characters


#### 10.12 Update
- making the prompt way simpler -> removing examples, making prompt slim

#### 17.12 Update
- Lets repeat the experiments but this time, lets create the ood-samples per-checkpoint. This should more closely follow the preference of the model.


In [ ]:
import os
import pandas as pd
from utils import load_rollouts, sample

In [ ]:
RELOAD=False
if RELOAD:
    rollouts : list[pd.DataFrame] = load_rollouts()
    with open('rollouts.parquet', 'wb') as file:
        pickle.dump(obj=rollouts, file=file)
else:
    with open('rollouts.parquet', 'rb') as file:
        rollouts = pickle.load(file)

In [ ]:
correct_rollouts = [sample(df, only_filter='positive') for df in rollouts]

In [4]:
correct = pd.concat(correct_rollouts, axis=0)

### OOD ready-to-augment dataset creation

Let's augment ground truths to contain a behavior. This may be a calculation error or a validation behavior.

In [5]:
def inject_calculation_error_prompt(row : pd.DataFrame) -> list[dict]:
    system = (
        "You are a helpful case generator and an expert in mathematical reasoning. You help with augmenting text in the way the user specifies."
    )
    prefix = (
        "You're given a question and a correct student answer, your task is to inject an arithmetic or algebraic error "
        "(e.g., adding, subtracting, multiplying, or simplifying incorrectly) "
        "if and only if the answer allows for a reasonable augmentation. IMPORTANT: Stay as close as possible to the correct answer "
        "and refrain from explicitly stating errors in the augmented response, e.g. writing 'I incorretly calculate ...' or '... (this is a calculation error) ...'."
        "In case it is unreasonable to augment the correct answer with an arithmetic or algebraic error (some answers do not contain arithmetic operations or algebraic manipulations), "
        "just return '#### Not applicable'. Otherwise return the complete error-augmented answer, pre-pended by a '####'. "
        "Abstract example: If the Correct Answer is [reasoning] [correct arithmetic step 1] ... [correct arithmetic step k] ... [correct result], then your answer should be " 
        "#### [reasoning] [correct arithmetic step 1] ... [incorrect arithmetic step k] ... [incorrect result], i.e. staying as close as possible to the initial answer, "
        "whilst still introducing a subtle error into the calculation. "
    )
    
    remove_prefix, remove_suffix = "system\nYou are a helpful assistant.\nuser\n", " Let's think step by step and output the final answer within \\boxed{}.\nassistant\n"
    question = row['input'].removeprefix(remove_prefix).removesuffix(remove_suffix)
    gt = row['output']
    prompt = [{
        'role' : 'system',
        'content' : system
    },
    {
        'role' : 'user',
        'content' : f"{prefix}\n\n\nQuestion:\n{question}\nSolution:\n{gt}\n"
    }]
    return prompt

In [6]:
calculation_error_prompts = pd.DataFrame({
    'prompt' : correct.apply(inject_calculation_error_prompt, axis=1),
    'step' : correct['step'],
    'old_index' : correct.index
})

In [7]:
calculation_error_prompts.iloc[0]['prompt'][1]['content']

'You\'re given a question and a correct student answer, your task is to inject an arithmetic or algebraic error (e.g., adding, subtracting, multiplying, or simplifying incorrectly) if and only if the answer allows for a reasonable augmentation. IMPORTANT: Stay as close as possible to the correct answer and refrain from explicitly stating errors in the augmented response, e.g. writing \'I incorretly calculate ...\' or \'... (this is a calculation error) ...\'.In case it is unreasonable to augment the correct answer with an arithmetic or algebraic error (some answers do not contain arithmetic operations or algebraic manipulations), just return \'#### Not applicable\'. Otherwise return the complete error-augmented answer, pre-pended by a \'####\'. Abstract example: If the Correct Answer is [reasoning] [correct arithmetic step 1] ... [correct arithmetic step k] ... [correct result], then your answer should be #### [reasoning] [correct arithmetic step 1] ... [incorrect arithmetic step k] ..

In [ ]:
with open('/u/rfechner/data/ariadne/ood-prompts-per-checkpoint.parquet', 'wb') as file:
    calculation_error_prompts.to_parquet(file)

In [8]:
def inject_validation_prompt(row : pd.DataFrame) -> list[dict]:
    system = (
        "You are a helpful case generator and an expert in mathematical reasoning. You help with augmenting text in the way the user specifies."
    )
    prefix = (
        "You're given a question and a correct student answer, your task is to inject verification behaviour "
        "(e.g., adding verification of arithmetic operations or a sanity check) "
        "if and only if the answer allows for a reasonable augmentation. IMPORTANT: Stay as close as possible to the correct answer. "
        "In case it is unreasonable to augment the correct answer with a verification, "
        "just return '#### Not applicable'. Otherwise return the complete verification-augmented answer, pre-pended by a '####'. "
        "Abstract example: If the Correct Answer is [reasoning] [solve step 1] ... [solve step k] ... [result], then your answer should be " 
        "#### [reasoning] [solve step 1] ... [solve step k] ... [verification] [result], i.e. staying as close as possible to the initial answer, "
        "whilst introducing a final verification. "
    )
    
    remove_prefix, remove_suffix = "system\nYou are a helpful assistant.\nuser\n", " Let's think step by step and output the final answer within \\boxed{}.\nassistant\n"
    question = row['input'].removeprefix(remove_prefix).removesuffix(remove_suffix)
    gt = row['output']
    prompt = [{
        'role' : 'system',
        'content' : system
    },
    {
        'role' : 'user',
        'content' : f"{prefix}\n\n\nQuestion:\n{question}\nSolution:\n{gt}\n"
    }]
    return prompt

In [9]:
validation_prompts = pd.DataFrame({
    'prompt' : correct.apply(inject_validation_prompt, axis=1),
    'step' : correct['step'],
    'old_index' : correct.index
})

In [10]:
validation_prompts.iloc[0]['prompt'][1]['content']

'You\'re given a question and a correct student answer, your task is to inject verification behaviour (e.g., adding verification of arithmetic operations or a sanity check) if and only if the answer allows for a reasonable augmentation. IMPORTANT: Stay as close as possible to the correct answer. In case it is unreasonable to augment the correct answer with a verification, just return \'#### Not applicable\'. Otherwise return the complete verification-augmented answer, pre-pended by a \'####\'. Abstract example: If the Correct Answer is [reasoning] [solve step 1] ... [solve step k] ... [result], then your answer should be #### [reasoning] [solve step 1] ... [solve step k] ... [verification] [result], i.e. staying as close as possible to the initial answer, whilst introducing a final verification. \n\n\nQuestion:\n$\\overline{BC}$ is parallel to the segment through $A$, and $AB = BC$. What is the number of degrees represented by $x$?\n\n[asy]\ndraw((0,0)--(10,0));\ndraw((0,3)--(10,3));\n

In [ ]:
with open('/u/rfechner/data/ariadne/ood-prompts-per-checkpoint-verify.parquet', 'wb') as file:
    validation_prompts.to_parquet(file)